In [10]:
# import packages 
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 

from scipy import stats
import statsmodels.api as sm

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
import dtreeviz

In [2]:
# read csv file 

heart_df = pd.read_csv("/Users/bera/Desktop/data/cardio_train.csv", delimiter =";")

heart_df

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,99993,19240,2,168,76.0,120,80,1,1,1,0,1,0
69996,99995,22601,1,158,126.0,140,90,2,2,0,0,1,1
69997,99996,19066,2,183,105.0,180,90,3,1,0,1,0,1
69998,99998,22431,1,163,72.0,135,80,1,2,0,0,0,1


#### There is no missing data, refer to analysis.ipynb

## Fix Data 

In [3]:
average_days_per_year = 365.25

heart_df['age'] = (heart_df['age']/ average_days_per_year).round().astype(int)

heart_df

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,50,2,168,62.0,110,80,1,1,0,0,1,0
1,1,55,1,156,85.0,140,90,3,1,0,0,1,1
2,2,52,1,165,64.0,130,70,3,1,0,0,0,1
3,3,48,2,169,82.0,150,100,1,1,0,0,1,1
4,4,48,1,156,56.0,100,60,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,99993,53,2,168,76.0,120,80,1,1,1,0,1,0
69996,99995,62,1,158,126.0,140,90,2,2,0,0,1,1
69997,99996,52,2,183,105.0,180,90,3,1,0,1,0,1
69998,99998,61,1,163,72.0,135,80,1,2,0,0,0,1


In [4]:
heart_df["bmi"] = (heart_df["weight"]/((heart_df["height"]/100) **2)).round(2)

heart_df.pop('id')
heart_df.pop('height')
heart_df.pop('weight')

heart_df

,age,gender,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,bmi
0,50,2,110,80,1,1,0,0,1,0,21.97
1,55,1,140,90,3,1,0,0,1,1,34.93
2,52,1,130,70,3,1,0,0,0,1,23.51
3,48,2,150,100,1,1,0,0,1,1,28.71
4,48,1,100,60,1,1,0,0,0,0,23.01
...,...,...,...,...,...,...,...,...,...,...,...
69995,53,2,120,80,1,1,1,0,1,0,26.93
69996,62,1,140,90,2,2,0,0,1,1,50.47
69997,52,2,180,90,3,1,0,1,0,1,31.35
69998,61,1,135,80,1,2,0,0,0,1,27.10


In [5]:
heart_df= heart_df[["age",
                    "gender",
                    "bmi",
                    "ap_hi",
                    "ap_lo",
                    "cholesterol",
                    "gluc",
                    "smoke",
                    "alco",
                    "active",
                    "cardio"]]
heart_df

,age,gender,bmi,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,50,2,21.97,110,80,1,1,0,0,1,0
1,55,1,34.93,140,90,3,1,0,0,1,1
2,52,1,23.51,130,70,3,1,0,0,0,1
3,48,2,28.71,150,100,1,1,0,0,1,1
4,48,1,23.01,100,60,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
69995,53,2,26.93,120,80,1,1,1,0,1,0
69996,62,1,50.47,140,90,2,2,0,0,1,1
69997,52,2,31.35,180,90,3,1,0,1,0,1
69998,61,1,27.10,135,80,1,2,0,0,0,1


## Train Decision Tree Model

In [14]:
y = heart_df["cardio"]
X = heart_df.drop(columns=["cardio"])


In [17]:
def train_val_test(df):
    X_train, X_temp, y_train, y_temp = train_test_split(X,
                                                        y,
                                                        train_size=0.7,
                                                        random_state=12)
    X_val, X_test, y_val, y_test = train_test_split(X_temp,
                                                    y_temp,
                                                    test_size=0.15,
                                                    random_state=12)
    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = train_val_test(heart_df)

In [32]:
# Check how stratified the data is. 
print(f"Y Train Data:\n{pd.Series(y_train).value_counts()/y_train.shape}")
print(f"\nY Test Data:\n{pd.Series(y_test).value_counts()/y_test.shape}")


Y Train Data:
cardio
1    0.500592
0    0.499408
Name: count, dtype: float64

Y Test Data:
cardio
0    0.509524
1    0.490476
Name: count, dtype: float64


https://medium.com/@masadeghi6/how-to-split-your-data-for-machine-learning-eae893a8799c#:~:text=split%20the%20data%20randomly%20into,of%20each%20architecture%20is%20maximized